In [11]:
from cellgrn.main import normalzie_rna,parse_edges,compute_all_cells_grn,summarize_grn,format_celltype_grn,format_sample_grn
import numpy as np
import pandas as pd
import os
import anndata as ad
from scipy import sparse
import pickle



In [2]:


cell_meta = pd.read_csv("/home/shaliu_fu/multireg/cellGRN/data/melanoma/metadata.csv",index_col=0)

input_rna = ad.read_h5ad(f"/home/shaliu_fu/multireg/cellGRN/data/melanoma/Melanoma-cell_line-RNA-counts.h5ad")
input_atac = ad.read_h5ad(f"/home/shaliu_fu/multireg/cellGRN/data/melanoma/Melanoma-cell_line-ATAC-peaks.h5ad")

# cell_types = cell_meta['annotated_labels']

In [3]:
all_tf = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/db/all_hg_TF.txt")] 

In [4]:

input_gene = input_rna.var.index.values
input_peak = input_atac.var.index.values
input_tf = list(set(input_gene) & set(all_tf))
cell_types = cell_meta['cell_type']

In [ ]:


for soft in ['linger',"scenic2"]:

    input_df1 = pd.DataFrame(input_rna.X.toarray(),index=input_rna.obs.index.values,columns=input_rna.var.index.values)
    peak_rename = [i.replace("-",":",1) for i in input_atac.var.index.values]
    input_df2 = pd.DataFrame(input_atac.X.toarray(),index=input_atac.obs.index.values,columns=peak_rename)

    outdir = f"../output/res_melanoma_{soft}/"
    os.system(f"mkdir -p {outdir}")


    cand_df = pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN/data/melanoma/{soft}_grn.csv",header=0)

    input_genes = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/melanoma/{soft}_gene.txt")]
    input_peaks = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/melanoma/{soft}_peak.txt")]

    # input_peaks = [i.replace(":","-") for i in input_peaks]
    input_df1 = input_df1[input_genes]
    input_df2 = input_df2[input_peaks]


    rna_data1,rna_data2 = normalzie_rna(input_df1)
    atac_data = input_df2.copy()

    input_tfs = [tf for tf in input_tf if tf in input_genes]
    tf_data1 = rna_data1[input_tfs].copy()
    tf_data2 = rna_data2[input_tfs].copy()

    edges_idx,edges_name = parse_edges(cand_df, input_tfs, input_genes, input_peaks)


    # grn_raw = compute_all_cells_grn(input_df1[input_tfs], input_df1, atac_data,edges_idx, edges_name,
    #     input_tfs, input_genes, input_peaks)

    # grn_scale = compute_all_cells_grn(tf_data1, rna_data1, atac_data,edges_idx, edges_name,
        # input_tfs, input_genes, input_peaks)
    grn_scale2 = compute_all_cells_grn(tf_data2, rna_data2, atac_data,edges_idx, edges_name,
        input_tfs, input_genes, input_peaks)
        
    with open(f"{outdir}/{soft}_cell_grn.pkl", "wb") as f:
        pickle.dump(grn_scale2.copy(), f)

    sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)


    tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
    tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)
    tf_gene_res_scale2.to_csv(os.path.join(outdir, "tf_gene_sample_scale2.csv"), index=False)
    tf_peak_res_scale2.to_csv(os.path.join(outdir, "tf_peak_sample_scale2.csv"), index=False)
    gene_peak_res_scale2.to_csv(os.path.join(outdir, "gene_peak_sample_scale2.csv"), index=False)

    tf_gene_ct_res_scale2.to_csv(os.path.join(outdir, "tf_gene_celltype_scale2.csv"), index=False)
    tf_peak_ct_res_scale2.to_csv(os.path.join(outdir, "tf_peak_celltype_scale2.csv"), index=False)
    gene_peak_ct_res_scale2.to_csv(os.path.join(outdir, "gene_peak_celltype_scale2.csv"), index=False)